# Interrupts

Interrupts 让你能够在特定位置暂停 graph 执行, 并在继续之前等待外部输入。这就能支撑 human-in-the-loop 模式: 你需要外部输入才能继续。当某个 interrupt 被触发时, LangGraph 会用其 [persistence](https://docs.langchain.com/oss/langgraph/persistence) 层保存 graph state, 并无限期等待, 直到你恢复执行。

Interrupts 的运作方式, 是在 graph node 的任意位置调用 `interrupt()` 函数。该函数接受任意 JSON 可序列化的值, 并将其暴露给调用方。当你准备好继续时, 通过用 `Command` 重新调用 graph 来恢复执行, 这个值随后会成为 node 内部 `interrupt()` 调用的返回值。

与静态断点 (在特定 node 之前或之后暂停) 不同, interrupts 是**动态的**: 它们可以放在代码中的任意位置, 也可以根据你的应用逻辑设置条件。

- **Checkpointing 会记住你的位置:** checkpointer 会写入精确的 graph state, 因此你可以稍后恢复, 即使处于错误 state 也一样。
- **`thread_id` 就是你的指针:** 设置 `config={"configurable": {"thread_id": ...}}`, 告诉 checkpointer 要加载哪个 state。
- **Interrupt payload 通过 `stream.interrupts` 暴露:** 使用 [event streaming](https://docs.langchain.com/oss/langgraph/event-streaming) (`graph.stream_events(..., version="v3")`) 时, 你传给 `interrupt()` 的值会出现在 `stream.interrupts` 上, 并且当 run 为等待输入而暂停时, `stream.interrupted` 为 `True`。

你选择的 `thread_id` 实际上就是你的持久化游标。复用它就会恢复同一个 checkpoint; 使用新值则会开启一个全新的 thread, 其 state 为空。

In [1]:
# 环境准备: 从 .env 加载 API keys
from dotenv import load_dotenv

load_dotenv()  # 自动向上查找 src/oss/langgraph/.env

# ---- 统一导入 ----
import operator
import os
import sqlite3
import tempfile
from typing import Annotated, Any, Literal, Optional, TypedDict

from pydantic import BaseModel

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, SystemMessage, ToolMessage
from langchain.tools import tool

from langgraph.checkpoint.memory import InMemorySaver, MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

## 使用 `interrupt` 暂停

`interrupt` 函数会暂停 graph 执行并把一个值返回给调用方。当你在 node 中调用 `interrupt` 时, LangGraph 会保存当前的 graph state, 并等待你带着输入恢复执行。

要使用 `interrupt`, 你需要:
1. 一个 **checkpointer** 来持久化 graph state (生产环境请使用持久化 checkpointer)
2. config 中有一个 **thread ID**, 让 runtime 知道要从哪个 state 恢复
3. 在你想要暂停的位置调用 `interrupt()` (payload 必须是 JSON 可序列化的)

```python
from langgraph.types import interrupt

def approval_node(state: State):
    # Pause and ask for approval
    approved = interrupt("Do you approve this action?")

    # When you resume, Command(resume=...) returns that value here
    return {"approved": approved}
```

当你调用 `interrupt` 时, 会发生以下情况:

1. **Graph 执行被挂起**, 停在调用 `interrupt` 的确切位置
2. **State 被保存**, 由 checkpointer 完成, 以便稍后恢复执行, 生产环境中这应该是一个持久化 checkpointer (例如由数据库支撑)
3. **Value 被返回**给调用方: 使用 [event streaming](https://docs.langchain.com/oss/langgraph/event-streaming) (`graph.stream_events(..., version="v3")`) 时出现在 `stream.interrupts` 上, 使用默认的 `invoke()` API 时则位于 `__interrupt__` 下; 它可以是任意 JSON 可序列化的值 (字符串、对象、数组等)
4. **Graph 无限期等待**, 直到你带着响应恢复执行
5. **Response 被传回**到 node 中 (在你恢复时), 成为 `interrupt()` 调用的返回值

## 恢复 interrupts

在某个 interrupt 暂停执行之后, 你通过再次调用 graph 来恢复它, 并传入一个包含 resume 值的 `Command`。resume 值会被传回 `interrupt` 调用, 让 node 能够带着外部输入继续执行。

驱动一个可能 interrupt 的 graph, 推荐方式是 [event streaming](https://docs.langchain.com/oss/langgraph/event-streaming) — 它通过 `stream.interrupts` 与 `stream.interrupted` 暴露 interrupts, 并通过 `stream.output` 暴露最终 state。

```python
from langgraph.types import Command

# Initial run - hits the interrupt and pauses
# thread_id is the persistent pointer (stores a stable ID in production)
config = {"configurable": {"thread_id": "thread-1"}}
stream = graph.stream_events({"input": "data"}, config=config, version="v3")

# Drain the stream to drive the run; stream.output awaits the final state.
final = stream.output

# stream.interrupted is True when the run paused for human input, and
# stream.interrupts contains the payloads passed to interrupt().
if stream.interrupted:
    print(stream.interrupts)
    # > (Interrupt(value='Do you approve this action?'),)

# Resume with the human's response
# The resume payload becomes the return value of interrupt() inside the node
resumed = graph.stream_events(Command(resume=True), config=config, version="v3")
final = resumed.output
```

> **注意**
>
> 默认的 `graph.invoke(...)` API 仍然可用, 它会在 `result["__interrupt__"]` 下暴露 interrupts。当你不需要 stream 投影时可以使用它; 否则优先使用 `graph.stream_events(..., version="v3")`。

**关于恢复的要点:**

- 恢复时必须使用与 interrupt 发生时**相同的 thread ID**
- 传给 `Command(resume=...)` 的值会成为 `interrupt` 调用的返回值
- 恢复时, node 会从调用 `interrupt` 的那个 node 的开头重新开始, 因此 `interrupt` 之前的任何代码都会再次运行
- 你可以传入任意 JSON 可序列化的值作为 resume 值, 当你指定了带类型的 `response_schema` 时会受校验约束

> **注意**
>
> `Command(resume=...)` 是**唯一**一种被设计为 `invoke()`/`stream()`/`stream_events()` 输入的 `Command` 用法。其他 `Command` 参数 (`update`、`goto`、`graph`) 是用于[从 node 函数返回](https://docs.langchain.com/oss/langgraph/graph-api#command)的。不要把 `Command(update=...)` 作为输入来继续多轮对话 — 请改为传入一个普通的 input dict。

## 定义 interrupt response schema

`interrupt` 的 `response_schema` 参数描述了你恢复 graph 时预期输入的格式。客户端可以用这个 schema 渲染带类型的输入表单。

> **注意**
>
> Interrupt response schema 需要 `langgraph>=1.2.12`。

传入一个 Pydantic model 类、来自 `typing_extensions` 的 `TypedDict`, 或一个 dataclass 来校验 resume 值。LangGraph 会把这个类型转换为 JSON Schema, 并把它包含在 `Interrupt.response_schema` 中, 与 interrupt 的 `value` payload 分开。这个 schema 在返回给调用方的 interrupts 上可用, 也可在 checkpoint task interrupts 中获取。

这个示例使用的是 LangGraph interrupt 测试中的 approval schema:

In [2]:
class Decision(BaseModel):
    approved: bool
    note: str | None = None


class State(TypedDict):
    answer: Any


def node(state: State) -> State:
    answer = interrupt({"question": "approve?"}, response_schema=Decision)
    return {"answer": answer}


graph = (
    StateGraph(State)
    .add_node("node", node)
    .add_edge(START, "node")
    .compile(checkpointer=InMemorySaver())
)
config = {"configurable": {"thread_id": "1"}}

result = graph.invoke({"answer": None}, config)
pending = result["__interrupt__"][0]
assert pending.response_schema == Decision.model_json_schema()

result = graph.invoke(Command(resume={"approved": True}), config)
assert result["answer"] == Decision(approved=True)

带有类型化 schema 时, `interrupt()` 会返回校验后的对象: 一个 Pydantic model 实例、`TypedDict` 对应的字典, 或一个 dataclass 实例。Pydantic 的校验与强制转换规则会生效。无效输入会抛出 `pydantic.ValidationError`; 你可以用同一个 thread ID 通过修正后的 `Command(resume=...)` 重试。

你也可以传入一个 JSON Schema 字典作为 `response_schema`。LangGraph 会原样暴露该字典, 但**不会**依据它校验 resume 值。当你省略 `response_schema` 时, 它的值为 `None`, resume 值会在没有 schema 校验的情况下直接通过。

## 常见模式

interrupts 带来的关键能力, 是暂停执行并等待外部输入。这在各种用例中都很有用, 包括:

- [审批 workflow](#approve-or-reject): 在执行关键操作 (API 调用、数据库变更、金融交易) 之前暂停
- [处理多个 interrupts](#handling-multiple-interrupts): 在单次调用中恢复多个 interrupts 时, 把 interrupt ID 与 resume 值配对
- [审查与编辑](#review-and-edit-state): 在继续之前, 让人审查并修改 LLM 输出或 tool 调用
- [Interrupt tool 调用](#interrupts-in-tools): 在执行 tool 调用之前暂停, 以便在执行前审查并编辑该 tool 调用
- [校验人工输入](#validating-human-input): 在进入下一步之前暂停, 以校验人工输入

### 通过 streaming 处理 human-in-the-loop (HITL) interrupts

在构建带 human-in-the-loop workflow 的交互式 agent 时, 你可以使用 [event streaming](https://docs.langchain.com/oss/langgraph/event-streaming), 在处理 interrupts 的同时并发消费消息 chunk 与 state snapshot。

在循环中使用 `graph.stream_events(..., version="v3")` 返回的带类型投影, 直到 run 结束:
- 通过 `stream.messages` 逐 token 地 stream AI 响应
- 通过 `stream.values` 观察每一步的 state snapshot
- 通过 `stream.interrupted` 检测 interrupts, 并从 `stream.interrupts` 读取其 payload
- 通过用 `Command(resume=...)` 再次调用 `stream_events` 来恢复执行, 并重复直到 `stream.interrupted` 为 false

```python
from langgraph.types import Command

stream_input: dict | Command = initial_input

while True:
    stream = graph.stream_events(stream_input, config=config, version="v3")

    # Stream LLM message chunks (including any in subgraphs) as they arrive.
    for message in stream.messages:
        for token in message.text:
            display_streaming_content(token)

    # After the run finishes (or pauses), check for interrupts and resume.
    if not stream.interrupted:
        final_state = stream.output
        break

    interrupt_info = stream.interrupts[0].value
    user_response = get_user_input(interrupt_info)
    stream_input = Command(resume=user_response)
```

- **`stream.messages`**: chat model 的输出, 以 content block 形式呈现; 遍历每个 `message.text` 获取 token delta。对于嵌套 subgraph, 从 `stream.subgraphs[*].messages` 读取消息 chunk。
- **`stream.values`**: 每一步之后的完整 state snapshot
- **`stream.interrupted` / `stream.interrupts`**: 每次 run 之后, 检查 graph 是否暂停; 从 `stream.interrupts` 读取 payload
- **`Command(resume=...)`**: 作为下一次 `stream_events` 的输入传入以恢复; 循环直到 run 完成且不再 interrupt

### 处理多个 interrupts

当并行分支同时 interrupt 时 (例如扇出到多个各自调用 `interrupt()` 的 node), 你可能需要在单次调用中恢复多个 interrupts。
在单次调用中恢复多个 interrupts 时, 把每个 interrupt ID 映射到它的 resume 值。
这能确保每个响应在 runtime 中都配对到正确的 interrupt。

In [3]:
class State(TypedDict):
    vals: Annotated[list[str], operator.add]


def node_a(state):
    answer = interrupt("question_a")
    return {"vals": [f"a:{answer}"]}


def node_b(state):
    answer = interrupt("question_b")
    return {"vals": [f"b:{answer}"]}


graph = (
    StateGraph(State)
    .add_node("a", node_a)
    .add_node("b", node_b)
    .add_edge(START, "a")
    .add_edge(START, "b")
    .add_edge("a", END)
    .add_edge("b", END)
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# Step 1: stream events to drive the run; both parallel nodes hit interrupt() and pause
stream = graph.stream_events({"vals": []}, config, version="v3")
_ = stream.output  # drive the stream to completion
# stream.interrupts contains the pending Interrupt payloads
print(stream.interrupts)
# > (Interrupt(value='question_a', id='...'), Interrupt(value='question_b', id='...'))

# Step 2: resume all pending interrupts at once
resume_map = {
    i.id: f"answer for {i.value}" for i in stream.interrupts
}
resumed = graph.stream_events(Command(resume=resume_map), config, version="v3")

print("Final state:", resumed.output)
# Final state: {'vals': ['a:answer for question_a', 'b:answer for question_b']}

[Interrupt(value='question_a', id='c55e58fe6ef3b140ebc96fb0e354615e', response_schema=None), Interrupt(value='question_b', id='9eb2824c83639620684a0fc08aa4d939', response_schema=None)]
Final state: {'vals': ['a:answer for question_a', 'b:answer for question_b']}


C:\Users\Ashit\Documents\GitHub\LangChain-Docs\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
C:\Users\Ashit\Documents\GitHub\LangChain-Docs\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


### 审批或拒绝

interrupts 最常见的用途之一, 是在关键操作之前暂停并请求审批。例如, 你可能想让人来审批一次 API 调用、一次数据库变更, 或任何其他重要决策。

```python
from typing import Literal
from langgraph.types import interrupt, Command

def approval_node(state: State) -> Command[Literal["proceed", "cancel"]]:
    # Pause execution; payload shows up on stream.interrupts (with stream_events) or result["__interrupt__"] (with invoke)
    is_approved = interrupt({
        "question": "Do you want to proceed with this action?",
        "details": state["action_details"]
    })

    # Route based on the response
    if is_approved:
        return Command(goto="proceed")  # Runs after the resume payload is provided
    else:
        return Command(goto="cancel")
```

恢复 graph 时, 传入 `True` 表示审批通过, 传入 `False` 表示拒绝。完整示例如下:

```python
# To approve
graph.stream_events(Command(resume=True), config=config, version="v3").output

# To reject
graph.stream_events(Command(resume=False), config=config, version="v3").output
```

In [4]:
class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt(
        {
            "question": "Approve this action?",
            "details": state["action_details"],
        }
    )

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}


builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

# Use a more durable checkpointer in production
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-123"}}
initial = graph.stream_events(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
    version="v3",
)
_ = initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'question': ..., 'details': ...}),)

# Resume with the decision; True routes to proceed, False to cancel
resumed = graph.stream_events(Command(resume=True), config=config, version="v3")
print(resumed.output["status"])

[Interrupt(value={'question': 'Approve this action?', 'details': 'Transfer $500'}, id='44cd72bcbdf9eaaa72a700841b0388be', response_schema=None)]
approved


### 审查并编辑 state

有时你想让人在继续之前审查并编辑 graph state 的一部分。这对于纠正 LLM、补充缺失信息或做出调整都很有用。

```python
from langgraph.types import interrupt

def review_node(state: State):
    # Pause and show the current content for review (payload surfaces on stream.interrupts)
    edited_content = interrupt({
        "instruction": "Review and edit this content",
        "content": state["generated_text"]
    })

    # Update the state with the edited version
    return {"generated_text": edited_content}
```

恢复时, 提供编辑后的内容。完整示例如下:

```python
graph.stream_events(
    Command(resume="The edited and improved text"),  # Value becomes the return from interrupt()
    config=config,
    version="v3",
).output
```

In [5]:
class ReviewState(TypedDict):
    generated_text: str


def review_node(state: ReviewState):
    # Ask a reviewer to edit the generated content
    updated = interrupt(
        {
            "instruction": "Review and edit this content",
            "content": state["generated_text"],
        }
    )
    return {"generated_text": updated}


builder = StateGraph(ReviewState)
builder.add_node("review", review_node)
builder.add_edge(START, "review")
builder.add_edge("review", END)

checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "review-42"}}
initial = graph.stream_events(
    {"generated_text": "Initial draft"}, config=config, version="v3"
)
_ = initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'instruction': ..., 'content': ...}),)

# Resume with the edited text from the reviewer
final_state = graph.stream_events(
    Command(resume="Improved draft after review"),
    config=config,
    version="v3",
)
print(final_state.output["generated_text"])  # -> "Improved draft after review"

[Interrupt(value={'instruction': 'Review and edit this content', 'content': 'Initial draft'}, id='4667f110ac0bbc90269d186349ae3a9d', response_schema=None)]
Improved draft after review


### Tool 中的 interrupts

你也可以把 interrupts 直接放在 tool 函数内部。这样每当该 tool 被调用时, tool 自身就会暂停以等待审批, 并允许在执行 tool 调用之前对其进行人工审查与编辑。

首先, 定义一个使用 `interrupt` 的 tool:

In [6]:
@tool
def send_email(to: str, subject: str, body: str):
    """Send an email to a recipient."""

    # Pause before sending; payload surfaces on stream.interrupts when using event streaming
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Approve sending this email?"
    })

    if response.get("action") == "approve":
        # Resume value can override inputs before executing
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)
        return f"Email sent to {final_to} with subject '{final_subject}'"
    return "Email cancelled by user"

当你希望审批逻辑与 tool 本身放在一起、从而可以在 graph 的不同部分复用时, 这种方式很有用。LLM 可以自然地调用该 tool, 而 interrupt 会在该 tool 被调用时暂停执行, 让你能够审批、编辑或取消该操作。完整示例如下 (本示例使用 DeepSeek 的 chat model):

In [7]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


@tool
def send_email(to: str, subject: str, body: str):
    """Send an email to a recipient."""

    # Pause before sending; payload surfaces on stream.interrupts when using event streaming
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Approve sending this email?",
    })

    if response.get("action") == "approve":
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)

        # Actually send the email (your implementation here)
        print(f"[send_email] to={final_to} subject={final_subject} body={final_body}")
        return f"Email sent to {final_to}"

    return "Email cancelled by user"


model = init_chat_model("deepseek:deepseek-chat", temperature=0).bind_tools([send_email])
tools_by_name = {"send_email": send_email}


def agent_node(state: AgentState):
    # LLM may decide to call the tool; interrupt pauses before sending
    result = model.invoke(state["messages"])
    return {"messages": [result]}


def tool_node(state: AgentState):
    """Performs the tool call"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}


def should_continue(state: AgentState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""
    messages = state["messages"]
    last_message = messages[-1]

    if last_message.tool_calls:
        return "tool_node"
    return END


builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tool_node", tool_node)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_continue, ["tool_node", END])  # Routes to "tools" or END
builder.add_edge("tool_node", "agent")  # Loop back after tools

# 使用临时目录下的 sqlite 文件作为 checkpointer, 避免落盘的 .db 污染仓库, 同时保证可重复运行
db_dir = tempfile.mkdtemp()
checkpointer = SqliteSaver(
    sqlite3.connect(os.path.join(db_dir, "tool-approval.db"), check_same_thread=False)
)
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "email-workflow"}}
initial = graph.stream_events(
    {
        "messages": [
            {"role": "user", "content": "Send an email to alice@example.com about the meeting"}
        ]
    },
    config=config,
    version="v3",
)
initial.output  # drive the stream to completion
print(initial.interrupts)  # -> (Interrupt(value={'action': 'send_email', ...}),)

# Resume with approval and optionally edited arguments
resumed = graph.stream_events(
    Command(resume={"action": "approve", "subject": "Updated subject"}),
    config=config,
    version="v3",
)
print(resumed.output["messages"][-1])  # -> Tool result returned by send_email

[Interrupt(value={'action': 'send_email', 'to': 'alice@example.com', 'subject': 'Meeting', 'body': 'Hi Alice,\n\nJust reaching out about the meeting. Let me know what works for you.\n\nBest regards', 'message': 'Approve sending this email?'}, id='edbd9cdd114142e3c9e644cc143dcbd7', response_schema=None)]
[send_email] to=alice@example.com subject=Updated subject body=Hi Alice,

Just reaching out about the meeting. Let me know what works for you.

Best regards


content=[{'type': 'text', 'text': 'The email has been sent to alice@example.com with the subject "Meeting."\n\nI kept the message fairly generic since you didn\'t specify details. If you\'d like, let me know the date, time, location, or agenda and I can send a follow-up with the specifics.', 'index': 0}] additional_kwargs={} response_metadata={'model_provider': 'deepseek', 'finish_reason': 'stop', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'output_version': 'v1'} id='lc_run--01a0f84d-3341-7811-8e70-f47c7bfe54cb' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 419, 'output_tokens': 58, 'total_tokens': 477, 'input_token_details': {'cache_read': 256}}


### 校验人工输入

有时你需要校验人工输入的值, 并在无效时重新提示。推荐做法是**每次 node 调用只调用一次 `interrupt()`**, 从 node 返回时把错误信息存入 state, 并使用一条 **conditional edge** 回环到该 node, 直到收集到有效值。

> **注意**
>
> **避免在单个 node 内使用 `while True` + `interrupt()` 循环。** 因为每次恢复时 node 都会从头重新运行 (见 [Interrupts 的规则](#rules-of-interrupts)), 一个多次调用 `interrupt()` 的循环会导致每次恢复都重放之前的所有迭代: 第一次恢复重放 1 次迭代, 第二次重放 2 次, 依此类推。结果是循环体内的任何代码都会被指数级地重复执行。

正确的模式:

1. 把重新提示的问题存入 state (例如 `pending_question`)。
2. 在 node 中**恰好调用一次** `interrupt()`, 传入来自 state 的当前问题。
3. 如果答案无效, 返回更新后的 `pending_question`, 让下一次调用重新提示。
4. 使用 `add_conditional_edges` 回环到该 node, 直到收集到有效值。

In [8]:
class FormState(TypedDict):
    age: int | None
    pending_question: str | None


def get_age_node(state: FormState):
    question = state.get("pending_question") or "What is your age?"
    answer = interrupt(question)  # called exactly once per invocation
    if isinstance(answer, int) and answer > 0:
        return {"age": answer, "pending_question": None}
    return {"pending_question": f"'{answer}' is not a valid age. Please enter a positive number."}


def route(state: FormState):
    return END if state.get("age") is not None else "collect_age"


builder = StateGraph(FormState)
builder.add_node("collect_age", get_age_node)
builder.add_edge(START, "collect_age")
builder.add_conditional_edges("collect_age", route)

每次恢复都会恰好调用一次 `get_age_node`, 运行一次 `interrupt()` 调用, 然后退出。当答案无效时, conditional edge 会回环, 下一次 interrupt 会用更新后的问题重新提示。每次恢复中没有任何代码会运行超过一次。完整示例如下:

In [9]:
class FormState(TypedDict):
    age: int | None
    pending_question: str | None


def get_age_node(state: FormState):
    question = state.get("pending_question") or "What is your age?"
    answer = interrupt(question)  # called exactly once per node invocation
    print(f"I got {answer}")  # runs exactly once per resume
    if isinstance(answer, int) and answer > 0:
        return {"age": answer, "pending_question": None}
    return {"pending_question": f"'{answer}' is not a valid age. Please enter a positive number."}


def route(state: FormState):
    # Loop back to collect_age until we have a valid age
    return END if state.get("age") is not None else "collect_age"


builder = StateGraph(FormState)
builder.add_node("collect_age", get_age_node)
builder.add_edge(START, "collect_age")
builder.add_conditional_edges("collect_age", route)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "form-1"}}
first = graph.stream_events({"age": None, "pending_question": None}, config=config, version="v3")
_ = first.output  # drive the stream to completion
print(first.interrupts)  # -> (Interrupt(value='What is your age?', ...),)

# Provide invalid data; the node re-prompts via the conditional edge
retry = graph.stream_events(Command(resume="thirty"), config=config, version="v3")
_ = retry.output
print(retry.interrupts)  # -> (Interrupt(value="'thirty' is not a valid age...", ...),)

# Provide valid data; route() returns END and the graph finishes
final = graph.stream_events(Command(resume=30), config=config, version="v3")
print(final.output["age"])  # -> 30

[Interrupt(value='What is your age?', id='aa06d4630ec2c878db7e6631761e024e', response_schema=None)]
I got thirty
[Interrupt(value="'thirty' is not a valid age. Please enter a positive number.", id='0e730e3babf76a944c1d68e1f6673801', response_schema=None)]
I got 30
30


## Interrupts 的规则

当你在 node 中调用 `interrupt` 时, LangGraph 通过抛出一个异常来挂起执行, 该异常向 runtime 发出暂停的信号。这个异常会沿调用栈向上传播并被 runtime 捕获, runtime 随即通知 graph 保存当前 state 并等待外部输入。

当执行恢复时 (在你提供所请求的输入之后), runtime 会从头重新开始整个 node — 它不会从调用 `interrupt` 的确切那一行继续。这意味着在 `interrupt` 之前运行过的任何代码都会再次执行。因此, 使用 interrupts 时有一些重要规则需要遵守, 以确保它们按预期工作。

### 不要用 try/except 包裹 `interrupt` 调用

`interrupt` 在调用点暂停执行的方式, 是抛出一个特殊异常。如果你把 `interrupt` 调用包在 try/except 块里, 就会捕获这个异常, interrupt 也就无法被传回 graph。

* ✅ 把 `interrupt` 调用与容易出错的代码分开
* ✅ 在 try/except 块中使用具体的异常类型

```python
def node_a(state: State):
    # ✅ Good: interrupting first, then handling
    # error conditions separately
    interrupt("What's your name?")
    try:
        fetch_data()  # This can fail
    except Exception as e:
        print(e)
    return state
```

```python
def node_a(state: State):
    # ✅ Good: catching specific exception types
    # will not catch the interrupt exception
    try:
        name = interrupt("What's your name?")
        fetch_data()  # This can fail
    except NetworkException as e:
        print(e)
    return state
```

* 🔴 不要用裸 try/except 块包裹 `interrupt` 调用

```python
def node_a(state: State):
    # ❌ Bad: wrapping interrupt in bare try/except
    # will catch the interrupt exception
    try:
        interrupt("What's your name?")
    except Exception as e:
        print(e)
    return state
```

### 不要在 node 内重排 `interrupt` 调用

在单个 node 中使用多个 interrupts 很常见, 但如果不小心处理, 这可能导致意外行为。

当一个 node 包含多个 interrupt 调用时, LangGraph 会为该 node 对应的执行 task 维护一个专属的 resume 值列表。每当执行恢复时, 它都会从 node 的开头开始。对于遇到的每个 interrupt, LangGraph 会检查该 task 的 resume 列表中是否存在匹配的值。匹配是**严格基于索引的**, 因此 node 内 interrupt 调用的顺序很重要。

* ✅ 让 `interrupt` 调用在各次 node 执行之间保持一致

```python
def node_a(state: State):
    # ✅ Good: interrupt calls happen in the same order every time
    name = interrupt("What's your name?")
    age = interrupt("What's your age?")
    city = interrupt("What's your city?")

    return {
        "name": name,
        "age": age,
        "city": city
    }
```

* 🔴 不要在 node 内有条件地跳过 `interrupt` 调用
* 🔴 不要用在不同执行之间不确定的逻辑来循环 `interrupt` 调用, 包括 `while True` 校验循环。请改用 conditional edge (见 [校验人工输入](#validating-human-input))

```python
def node_a(state: State):
    # ❌ Bad: conditionally skipping interrupts changes the order
    name = interrupt("What's your name?")

    # On first run, this might skip the interrupt
    # On resume, it might not skip it - causing index mismatch
    if state.get("needs_age"):
        age = interrupt("What's your age?")

    city = interrupt("What's your city?")

    return {"name": name, "city": city}
```

```python
def node_a(state: State):
    # ❌ Bad: looping based on non-deterministic data
    # The number of interrupts changes between executions
    results = []
    for item in state.get("dynamic_list", []):  # List might change between runs
        result = interrupt(f"Approve {item}?")
        results.append(result)

    return {"results": results}
```

### 不要在 `interrupt` 调用中返回复杂值

取决于所用的 checkpointer, 复杂值可能无法序列化 (例如无法序列化函数)。为了让你的 graph 能适应任何部署环境, 最佳实践是只使用能够合理序列化的值。

* ✅ 向 `interrupt` 传入简单的、JSON 可序列化的类型
* ✅ 传入值为简单类型的字典/对象

```python
def node_a(state: State):
    # ✅ Good: passing simple types that are serializable
    name = interrupt("What's your name?")
    count = interrupt(42)
    approved = interrupt(True)

    return {"name": name, "count": count, "approved": approved}
```

```python
def node_a(state: State):
    # ✅ Good: passing dictionaries with simple values
    response = interrupt({
        "question": "Enter user details",
        "fields": ["name", "email", "age"],
        "current_values": state.get("user", {})
    })

    return {"user": response}
```

* 🔴 不要向 `interrupt` 传入函数、类实例或其他复杂对象

```python
def validate_input(value):
    return len(value) > 0

def node_a(state: State):
    # ❌ Bad: passing a function to interrupt
    # The function cannot be serialized
    response = interrupt({
        "question": "What's your name?",
        "validator": validate_input  # This will fail
    })
    return {"name": response}
```

```python
class DataProcessor:
    def __init__(self, config):
        self.config = config

def node_a(state: State):
    processor = DataProcessor({"mode": "strict"})

    # ❌ Bad: passing a class instance to interrupt
    # The instance cannot be serialized
    response = interrupt({
        "question": "Enter data to process",
        "processor": processor  # This will fail
    })
    return {"result": response}
```

### 在 `interrupt` 之前调用的副作用必须是幂等的

由于 interrupts 的工作方式是重新运行调用它们的 node, 因此在 `interrupt` 之前调用的副作用 (理想情况下) 应当是幂等的。作为背景, 幂等意味着同一个操作可以被应用多次, 而不会在首次执行之外改变结果。

举个例子, 你的 node 里可能有一次用于更新记录的 API 调用。如果在这次调用之后又调用了 `interrupt`, 那么当 node 恢复时它会多次重新运行, 可能会覆盖最初的更新或产生重复记录。

* ✅ 在 `interrupt` 之前使用幂等操作
* ✅ 把副作用放在 `interrupt` 调用之后
* ✅ 尽可能把副作用拆分到单独的 node 中

```python
def node_a(state: State):
    # ✅ Good: using upsert operation which is idempotent
    # Running this multiple times will have the same result
    db.upsert_user(
        user_id=state["user_id"],
        status="pending_approval"
    )

    approved = interrupt("Approve this change?")

    return {"approved": approved}
```

```python
def node_a(state: State):
    # ✅ Good: placing side effect after the interrupt
    # This ensures it only runs once after approval is received
    approved = interrupt("Approve this change?")

    if approved:
        db.create_audit_log(
            user_id=state["user_id"],
            action="approved"
        )

    return {"approved": approved}
```

```python
def approval_node(state: State):
    # ✅ Good: only handling the interrupt in this node
    approved = interrupt("Approve this change?")

    return {"approved": approved}

def notification_node(state: State):
    # ✅ Good: side effect happens in a separate node
    # This runs after approval, so it only executes once
    if (state.approved):
        send_notification(
            user_id=state["user_id"],
            status="approved"
        )

    return state
```

* 🔴 不要在 `interrupt` 之前执行非幂等操作
* 🔴 不要在未检查记录是否存在的情况下创建新记录

```python
def node_a(state: State):
    # ❌ Bad: creating a new record before interrupt
    # This will create duplicate records on each resume
    audit_id = db.create_audit_log({
        "user_id": state["user_id"],
        "action": "pending_approval",
        "timestamp": datetime.now()
    })

    approved = interrupt("Approve this change?")

    return {"approved": approved, "audit_id": audit_id}
```

```python
def node_a(state: State):
    # ❌ Bad: appending to a list before interrupt
    # This will add duplicate entries on each resume
    db.append_to_history(state["user_id"], "approval_requested")

    approved = interrupt("Approve this change?")

    return {"approved": approved}
```

## 与作为函数调用的 subgraph 配合使用

在 node 内调用 subgraph 时, 父 graph 会从调用该 subgraph 并触发 `interrupt` 的那个 **node 的开头**恢复执行。同样, **subgraph** 也会从调用 `interrupt` 的那个 node 的开头恢复。

```python
def node_in_parent_graph(state: State):
    some_code()  # <-- This will re-execute when resumed
    # Invoke a subgraph as a function.
    # The subgraph contains an `interrupt` call.
    subgraph_result = subgraph.invoke(some_input)
    # ...

def node_in_subgraph(state: State):
    some_other_code()  # <-- This will also re-execute when resumed
    result = interrupt("What's your name?")
    # ...
```

## 使用 interrupts 进行调试

为了调试和测试 graph, 你可以使用静态 interrupts 作为断点, 一次一个 node 地步进执行 graph。静态 interrupts 会在 node 执行之前或之后的确定位置触发。你可以在编译 graph 时通过指定 `interrupt_before` 与 `interrupt_after` 来设置它们。

> **注意**
>
> Static interrupts **不**推荐用于 human-in-the-loop workflow。请改用 `interrupt` 函数。

### 编译时

```python
graph = builder.compile(
    interrupt_before=["node_a"],  # [!code highlight]
    interrupt_after=["node_b", "node_c"],  # [!code highlight]
    checkpointer=checkpointer,
)

# Pass a thread ID to the graph
config = {
    "configurable": {
        "thread_id": "some_thread"
    }
}

# Run the graph until the breakpoint
graph.invoke(inputs, config=config)  # [!code highlight]

# Resume the graph
graph.invoke(None, config=config)  # [!code highlight]
```

1. 断点在 `compile` 时设置。
2. `interrupt_before` 指定应在 node 执行之前暂停执行的 node。
3. `interrupt_after` 指定应在 node 执行之后暂停执行的 node。
4. 启用断点需要一个 checkpointer。
5. graph 会运行到命中第一个断点为止。
6. 通过为 input 传入 `None` 来恢复 graph。这会让 graph 运行到命中下一个断点为止。

### 运行时

```python
config = {
    "configurable": {
        "thread_id": "some_thread"
    }
}

# Run the graph until the breakpoint
graph.invoke(
    inputs,
    interrupt_before=["node_a"],  # [!code highlight]
    interrupt_after=["node_b", "node_c"],  # [!code highlight]
    config=config,
)

# Resume the graph
graph.invoke(None, config=config)  # [!code highlight]
```

1. 调用 `graph.invoke` 时传入 `interrupt_before` 与 `interrupt_after` 参数。这是运行时配置, 每次调用都可以更改。
2. `interrupt_before` 指定应在 node 执行之前暂停执行的 node。
3. `interrupt_after` 指定应在 node 执行之后暂停执行的 node。
4. graph 会运行到命中第一个断点为止。
5. 通过为 input 传入 `None` 来恢复 graph。这会让 graph 运行到命中下一个断点为止。